# RQ3  Generation Quality: End-to-End Compliance Assessment Evaluation

**Research Question:** How accurately does the best-performing pipeline evaluate
ESRS compliance, and where does the system fail?

**Experimental design:** Single-condition characterisation study. The best RQ1
pipeline with ESRS context injected is evaluated
against the full Golden Standard (12 questions × 17 companies = 204 samples).
No paired comparison, no ablation — just: how well does this system work,
and where does it fail?

**Three evaluation layers:**

| Layer | What it measures | Primary metric |
|---|---|---|
| A. Deterministic metrics | Compliance label accuracy + citation page overlap | Macro-F1 |
| B. LLM-judge rubric (5 dimensions, 1-5 each) | Factual correctness, completeness, grounding, citation usefulness, compliance quality | Mean dimension scores |
| C. Error analysis | Per-standard breakdown, retrieval-generation gaps, confusion matrix | Qualitative |

**Key methodological decisions:**
- Generated answers are **reused** from RQ2's 'with_esrs_context'/no_esrs_context 
  condition — no new OpenAI generation calls.
- The LLM judge (Claude Sonnet 4.5) is used for evaluation 
- The generated compliance level is **withheld** from the judge to prevent
  anchoring bias (Zheng et al., 2024).
- All metrics are reported with **95% bootstrap confidence intervals**
  (10,000 resamples).
- A **stratified 20-sample manual audit subset** is exported as a CSV template
  for human calibration (no API calls required).

## 1. Setup and Imports

In [1]:
import sys, os, json, re, hashlib, warnings, logging, math
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, field
from statistics import mean, stdev
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix as sk_confusion_matrix,
    cohen_kappa_score, precision_recall_fscore_support,
)

warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(level=logging.WARNING)

# --- Project root and path setup ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# --- Parse .env (comment-tolerant, no python-dotenv dependency) ---
_env_path = PROJECT_ROOT / ".env"
if _env_path.exists():
    for _line in _env_path.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _, _v = _line.partition("=")
            _k = _k.strip()
            if _k:
                os.environ.setdefault(_k, _v.strip().strip('"').strip("'"))

from rag.config.settings import (
    GOLDEN_STANDARD_PATH, RQ2_RESULTS_DIR, RQ3_RESULTS_DIR,
    BEST_PIPELINE_PATH, ensure_dirs,
)

ensure_dirs()

# ---------------------------------------------------------------------------
# RQ2 → RQ3 Configuration
# ---------------------------------------------------------------------------
# RQ2 finding: did ESRS context injection produce a significant improvement?
# Set to True if RQ2 showed significant benefit; False if null result.
# This boolean controls which RQ2 condition RQ3 evaluates:
#   False → no_esrs_context  (ESRS context had no significant effect)
#   True  → full_system      (ESRS context significantly helped)
ESRS_CONTEXT_HELPS = False

# Select the RQ2 condition based on the RQ2 finding
RQ3_CONDITION = "full_system" if ESRS_CONTEXT_HELPS else "no_esrs_context"
NB_CACHE_DIR = RQ2_RESULTS_DIR / "nb_cache"

# --- Shared constants ---
LABELS = ["Low", "Medium", "High"]
LABEL_ORD = {"Low": 0, "Medium": 1, "High": 2}

# --- Publication-quality plot defaults ---
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12,
    "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
})
# Colorblind-friendly palette (Okabe-Ito inspired)
CLR_PRIMARY = "#0072B2"   # blue
CLR_ACCENT = "#D55E00"    # vermillion
CLR_SUCCESS = "#009E73"   # green
CLR_WARN = "#E69F00"      # amber
CLR_GREY = "#999999"      # grey

print(f"Project root  : {PROJECT_ROOT}")
print(f"RQ2 cache dir : {NB_CACHE_DIR}")
print(f"RQ3 results   : {RQ3_RESULTS_DIR}")
print(f"ESRS context helps (RQ2 finding): {ESRS_CONTEXT_HELPS}")
print(f"RQ3 condition : {RQ3_CONDITION}")

Project root  : /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot
RQ2 cache dir : /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq2/nb_cache
RQ3 results   : /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq3
ESRS context helps (RQ2 finding): False
RQ3 condition : no_esrs_context


## 2. Load Golden Standard and RQ2 Generated Answers

RQ3 reuses the generated answers from RQ2. Which condition is evaluated
depends on the `ESRS_CONTEXT_HELPS` boolean above:
- If **False** (null result) → uses `no_esrs_context` responses
- If **True** → uses `full_system` (with ESRS context) responses

Each of the 204 samples has a cached JSON file in the RQ2 `nb_cache`
directory. We match golden standard samples to their cached responses
using the same content-hash cache key that the RQ2 notebook used.

In [2]:
# --- Load golden standard ---
raw_golden = json.loads(GOLDEN_STANDARD_PATH.read_text(encoding="utf-8"))

# Build a flat list of all 204 samples with their gold fields
samples = []
for q in raw_golden["questions"]:
    for c in q["per_company"]:
        # Extract gold evidence page numbers for citation metrics
        gold_pages = []
        for ev in c.get("evidence", []):
            pn = str(ev.get("page_number", "")).strip()
            if pn:
                gold_pages.append(pn)
        samples.append({
            "question_id": q["id"],
            "question": q["question"],
            "esrs_standard": q.get("esrs_standard", ""),
            "esrs_disclosure_id": q.get("esrs_disclosure_id", ""),
            "company": c["company"],
            "ground_truth_answer": c.get("ground_truth_answer", ""),
            "fulfillment_level": c.get("fulfillment_level", ""),
            "fulfillment_rationale": c.get("fulfillment_rationale", ""),
            "gold_evidence_pages": gold_pages,
        })

print(f"Golden standard samples loaded: {len(samples)}")
print(f"Questions: {len(raw_golden['questions'])}, Companies per question: {len(samples) // len(raw_golden['questions'])}")

Golden standard samples loaded: 204
Questions: 12, Companies per question: 17


In [3]:
# --- ESRS context lengths per question (from RQ2 generation run) ---
# These len(esrs_context_text) values are needed to reconstruct the
# cache key that maps each sample to its cached generated response.
# For the full_system condition, the actual context length is used.
# For the no_esrs_context condition, the context length is always 0.
ESRS_CTX_LENGTHS = {
    "Q1": 14309, "Q2": 2517, "Q3": 10595, "Q4": 9590,
    "Q5": 4009, "Q6": 25453, "Q7": 9480, "Q8": 13080,
    "Q9": 19566, "Q10": 18847, "Q11": 8089, "Q12": 9763,
}

def _cache_key(question, company, condition, esrs_context_len):
    """Reconstruct the RQ2 nb_cache hash key for a given sample."""
    blob = f"{question}|{company}|{condition}|{esrs_context_len}"
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

def _get_ctx_len(question_id):
    """Return the ESRS context length appropriate for the current RQ3 condition.
    
    full_system    → actual context length (ESRS text was injected)
    no_esrs_context → 0 (no ESRS text was injected)
    """
    if RQ3_CONDITION == "full_system":
        return ESRS_CTX_LENGTHS.get(question_id, 0)
    return 0

# Index all cache files by their hash stem for O(1) lookup
cache_index = {}
cdir = NB_CACHE_DIR / RQ3_CONDITION
if cdir.exists():
    for fp in cdir.glob("*.json"):
        cache_index[fp.stem] = fp

print(f"Cache files indexed: {len(cache_index)} (condition: {RQ3_CONDITION})")

Cache files indexed: 204 (condition: no_esrs_context)


In [4]:
# --- Match each golden sample to its cached RQ2 generated answer ---

def _extract_gen(cache_entry):
    """Extract structured answer fields from a cached RQ2 response.
    
    The cache stores fields at the top level (compliance_level, rationale)
    and also in a raw_response JSON string. We prefer raw_response for
    canonical field name consistency.
    """
    raw = cache_entry.get("raw_response")
    if raw and isinstance(raw, str):
        try:
            parsed = json.loads(raw)
            return {
                "compliance_level": parsed.get("compliance_level", cache_entry.get("compliance_level", "")),
                "direct_answer": parsed.get("direct_answer", cache_entry.get("answer", "")),
                "compliance_rationale": parsed.get("compliance_rationale", cache_entry.get("rationale", "")),
                "key_findings": parsed.get("key_findings", cache_entry.get("key_findings", [])),
                "disclosure_gaps": parsed.get("disclosure_gaps", cache_entry.get("disclosure_gaps", [])),
                "cited_pages": parsed.get("cited_pages", cache_entry.get("cited_pages", [])),
            }
        except (json.JSONDecodeError, TypeError):
            pass
    # Fallback to top-level keys
    return {
        "compliance_level": cache_entry.get("compliance_level", ""),
        "direct_answer": cache_entry.get("answer", ""),
        "compliance_rationale": cache_entry.get("rationale", ""),
        "key_findings": cache_entry.get("key_findings", []),
        "disclosure_gaps": cache_entry.get("disclosure_gaps", []),
        "cited_pages": cache_entry.get("cited_pages", []),
    }

def _norm_level(v):
    return {"low": "Low", "medium": "Medium", "high": "High"}.get(str(v).strip().lower(), v)

# Build the RQ3 evaluation table
rq3_samples = []  # fully matched samples with gold + generated fields
missing = []

for s in samples:
    qid = s["question_id"]
    ctx_len = _get_ctx_len(qid)  # condition-aware context length
    hk = _cache_key(s["question"], s["company"], RQ3_CONDITION, ctx_len)
    fp = cache_index.get(hk)
    
    if fp is None:
        missing.append((qid, s["company"]))
        continue
    
    cache_data = json.loads(fp.read_text("utf-8"))
    gen = _extract_gen(cache_data)
    
    rq3_samples.append({
        # Gold fields
        "question_id": qid,
        "question": s["question"],
        "esrs_standard": s["esrs_standard"],
        "esrs_disclosure_id": s["esrs_disclosure_id"],
        "company": s["company"],
        "gold_level": s["fulfillment_level"],
        "gold_rationale": s["fulfillment_rationale"],
        "gold_answer": s["ground_truth_answer"],
        "gold_evidence_pages": s["gold_evidence_pages"],
        # Generated fields
        "pred_level": _norm_level(gen["compliance_level"]),
        "gen_direct_answer": gen["direct_answer"],
        "gen_rationale": gen["compliance_rationale"],
        "gen_key_findings": gen["key_findings"],
        "gen_disclosure_gaps": gen["disclosure_gaps"],
        "gen_cited_pages": gen["cited_pages"],
    })

print(f"Matched RQ3 samples: {len(rq3_samples)} / {len(samples)}")
if missing:
    print(f"Missing cache entries: {len(missing)} (first 3: {missing[:3]})")
    print("WARNING: Some samples could not be matched. Check ESRS_CTX_LENGTHS.")

Matched RQ3 samples: 204 / 204


## 3. Load RQ1 Retrieval Results (for Gap Analysis)

The retrieval-generation gap analysis (Section 8) needs Recall@15 per sample
from the best RQ1 pipeline. We load this now so it is available for later for the analysis in section 8.

In [5]:
# --- Load per-sample Recall@15 from RQ1 best pipeline ---
RQ1_DETAILS_DIR = PROJECT_ROOT / "data" / "artifacts" / "evaluation_results" / "rq1" / "config_details"
rq1_recall = {}  # (question_id, company) -> recall@15

rq1_path = RQ1_DETAILS_DIR / "fixed_1024_hybrid_rerank.json"
if rq1_path.exists():
    rq1_data = json.loads(rq1_path.read_text("utf-8"))
    for ps in rq1_data.get("details", {}).get("per_sample_metrics", []):
        rq1_recall[(ps["question_id"], ps["company"])] = ps.get("recall@15", None)
    print(f"RQ1 Recall@15 loaded for {len(rq1_recall)} samples")
else:
    print(f"WARNING: RQ1 details not found at {rq1_path}")

# Attach recall@15 to each RQ3 sample
for s in rq3_samples:
    s["recall_at_15"] = rq1_recall.get((s["question_id"], s["company"]), None)

RQ1 Recall@15 loaded for 204 samples


## 4. Approach A — Deterministic Compliance-Level Metrics

Evaluate how well the pipeline predicts the gold Low / Medium / High
compliance level. These are the same classification metrics used in RQ2
but reported for a single condition with bootstrap confidence intervals.

**Metrics:**
- **Macro-F1**: arithmetic mean of per-class F1
- **Balanced accuracy**: mean per-class recall
- **Exact accuracy**: fraction of exact label matches
- **Weighted Cohen's kappa**: ordinal agreement with linear weights
- **Off-by-one accuracy**: prediction within one ordinal step
- **Mean ordinal error**: mean |gold_ord - pred_ord| on 0-2 scale

In [6]:
from rag.evaluation.rq3_metrics import (
    normalize_label, ordinal_error, compliance_metrics,
    normalize_page_numbers, citation_page_metrics,
    bootstrap_ci, bootstrap_metric_ci,
)

# --- Extract gold and predicted label lists ---
gold_labels = [s["gold_level"] for s in rq3_samples if s["gold_level"] in LABELS]
pred_labels = [s["pred_level"] for s in rq3_samples if s["gold_level"] in LABELS]

# Filter to valid predictions only
valid_pairs = [(g, p) for g, p in zip(gold_labels, pred_labels) if p in LABELS]
gold_valid = [g for g, _ in valid_pairs]
pred_valid = [p for _, p in valid_pairs]

print(f"Samples with valid gold+pred labels: {len(valid_pairs)} / {len(rq3_samples)}")
print(f"Gold distribution: {dict(Counter(gold_valid))}")
print(f"Pred distribution: {dict(Counter(pred_valid))}")

# --- Compute aggregate metrics ---
det_metrics = compliance_metrics(gold_valid, pred_valid)

# --- Print results ---
print(f"\n{'Metric':<30s} {'Value':>10s}")
print("─" * 42)
for metric_name in ["macro_f1", "balanced_accuracy", "exact_accuracy",
                     "weighted_kappa", "off_by_one_accuracy", "mean_ordinal_error"]:
    val = det_metrics.get(metric_name, float("nan"))
    print(f"{metric_name:<30s} {val:>10.4f}")

Samples with valid gold+pred labels: 204 / 204
Gold distribution: {'High': 145, 'Low': 26, 'Medium': 33}
Pred distribution: {'High': 132, 'Medium': 66, 'Low': 6}

Metric                              Value
──────────────────────────────────────────
macro_f1                           0.5451
balanced_accuracy                  0.5589
exact_accuracy                     0.6912
weighted_kappa                     0.4075
off_by_one_accuracy                0.9608
mean_ordinal_error                 0.3480


### 4.1 Bootstrap Confidence Intervals for Compliance Metrics

Since RQ3 is a single-condition study (no paired comparison), we report
95% bootstrap CIs (10,000 resamples) instead of paired statistical tests.

In [7]:
# --- Bootstrap CIs for key classification metrics ---
def _macro_f1(g, p):
    return f1_score(g, p, labels=LABELS, average="macro", zero_division=0)

def _bal_acc(g, p):
    recalls = []
    for lbl in LABELS:
        pos = sum(1 for v in g if v == lbl)
        hits = sum(1 for gi, pi in zip(g, p) if gi == lbl and pi == lbl)
        recalls.append(hits / pos if pos else 0.0)
    return mean(recalls)

def _kappa(g, p):
    if len(set(g)) <= 1:
        return 1.0 if g == p else 0.0
    return cohen_kappa_score(g, p, labels=LABELS, weights="linear")

boot_results = {}
for name, fn in [("Macro-F1", _macro_f1), ("Balanced Accuracy", _bal_acc),
                  ("Weighted Kappa", _kappa)]:
    ci = bootstrap_metric_ci(gold_valid, pred_valid, fn)
    boot_results[name] = ci
    print(f"{name:<25s}  {ci['mean']:.4f}  [{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]")

Macro-F1                   0.5451  [0.4481, 0.6323]
Balanced Accuracy          0.5589  [0.4774, 0.6402]
Weighted Kappa             0.4075  [0.2852, 0.5216]


## 5. Citation Page Overlap Metrics

Compare the page numbers cited in the generated answer against the gold
evidence page numbers. This measures whether the system cites the right
pages from the sustainability reports.

Page numbers are normalised to handle format variations (e.g., `p.12`,
`42`, `43-44`, `Evidence 3 (p.150)`).

In [8]:
# --- Compute citation page metrics for each sample ---
cite_precisions = []
cite_recalls = []
cite_f1s = []
n_gold_has_pages = 0
n_pred_has_pages = 0

for s in rq3_samples:
    gold_pages = normalize_page_numbers(s["gold_evidence_pages"])
    pred_pages = normalize_page_numbers(s["gen_cited_pages"])
    
    if gold_pages:
        n_gold_has_pages += 1
    if pred_pages:
        n_pred_has_pages += 1
    
    metrics = citation_page_metrics(gold_pages, pred_pages)
    s["cite_precision"] = metrics["precision"]
    s["cite_recall"] = metrics["recall"]
    s["cite_f1"] = metrics["f1"]
    s["gold_page_set"] = gold_pages
    s["pred_page_set"] = pred_pages
    
    cite_precisions.append(metrics["precision"])
    cite_recalls.append(metrics["recall"])
    cite_f1s.append(metrics["f1"])

# --- Report aggregate citation metrics ---
print(f"Samples with gold pages: {n_gold_has_pages} / {len(rq3_samples)}")
print(f"Samples with predicted pages: {n_pred_has_pages} / {len(rq3_samples)}")
print()

cite_prec_ci = bootstrap_ci(cite_precisions)
cite_rec_ci = bootstrap_ci(cite_recalls)
cite_f1_ci = bootstrap_ci(cite_f1s)

print(f"{'Metric':<25s}  {'Mean':>7s}  {'95% CI':>20s}")
print("─" * 55)
for name, ci in [("Citation Precision", cite_prec_ci),
                  ("Citation Recall", cite_rec_ci),
                  ("Citation F1", cite_f1_ci)]:
    print(f"{name:<25s}  {ci['mean']:>7.4f}  [{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]")

Samples with gold pages: 198 / 204
Samples with predicted pages: 204 / 204

Metric                        Mean                95% CI
───────────────────────────────────────────────────────
Citation Precision          0.1217  [0.0936, 0.1528]
Citation Recall             0.2614  [0.2092, 0.3149]
Citation F1                 0.1498  [0.1175, 0.1841]


## 6. Approach B — LLM-as-Judge (Multi-Dimensional Rubric)

Each generated answer is evaluated by Claude Sonnet 4.5 across five
quality dimensions (1-5 scale each) in a **single API call per sample**.

**Dimensions:**
1. Factual correctness
2. Completeness
3. Grounding faithfulness
4. Citation usefulness
5. Compliance assessment quality

The generated compliance level is **withheld** from the judge prompt to
prevent anchoring bias. All results are cached to disk — if a sample is
already cached, the API call is skipped.

In [9]:
from rag.evaluation.rq3_judge import (
    RQ3JudgeInput, RQ3JudgeOutput, judge_sample, run_judge_batch,
    get_judge_version, JUDGE_CACHE_DIR, DIMENSION_NAMES,
)

print(f"RQ3 judge version: {get_judge_version()}")
print(f"Judge cache dir  : {JUDGE_CACHE_DIR}")
print(f"Dimensions       : {', '.join(DIMENSION_NAMES)}")

RQ3 judge version: rq3_v1_340948a222
Judge cache dir  : /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq3/judge_cache
Dimensions       : factual_correctness, completeness, grounding_faithfulness, citation_usefulness, compliance_assessment_quality


In [10]:
# --- Build judge inputs for all samples ---
judge_inputs = []
for s in rq3_samples:
    judge_inputs.append(RQ3JudgeInput(
        question=s["question"],
        esrs_standard=s["esrs_standard"],
        esrs_disclosure_id=s["esrs_disclosure_id"],
        company=s["company"],
        gold_fulfillment_level=s["gold_level"],
        gold_fulfillment_rationale=s["gold_rationale"],
        gold_ground_truth_answer=s["gold_answer"],
        generated_direct_answer=s["gen_direct_answer"],
        generated_compliance_rationale=s["gen_rationale"],
        generated_key_findings=s["gen_key_findings"],
        generated_disclosure_gaps=s["gen_disclosure_gaps"],
        generated_cited_pages=s["gen_cited_pages"],
    ))

print(f"Judge inputs prepared: {len(judge_inputs)}")

Judge inputs prepared: 204


In [11]:
# --- Run the judge batch (cached results are loaded, uncached call API) ---
# Set dry_run=True to see how many are cached without making API calls
judge_results = run_judge_batch(judge_inputs, dry_run=False)

n_cached = sum(1 for _, was_cached in judge_results if was_cached)
n_api = len(judge_results) - n_cached
print(f"\nJudge results: {len(judge_results)} total, {n_cached} cached, {n_api} API calls")

# Attach judge scores to each sample
for s, (judge_out, _) in zip(rq3_samples, judge_results):
    s["judge"] = judge_out

/Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.anthropic.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.anthropic.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/.venv/lib/python3.12/site-packages/urllib3


Judge results: 204 total, 2 cached, 202 API calls


### 6.1 Aggregate Judge Scores with Bootstrap CIs

Report the mean score for each dimension and the overall mean across
all five dimensions, each with 95% bootstrap confidence intervals.

In [12]:
# --- Aggregate judge scores per dimension ---
dim_scores = {dim: [] for dim in DIMENSION_NAMES}
mean_scores = []

for s in rq3_samples:
    j = s["judge"]
    for dim in DIMENSION_NAMES:
        dim_scores[dim].append(getattr(j, dim))
    mean_scores.append(j.mean_score)

# --- Print dimension-level results with bootstrap CIs ---
print(f"{'Dimension':<35s}  {'Mean':>6s}  {'95% CI':>20s}  {'Median':>6s}")
print("─" * 75)

dim_cis = {}
for dim in DIMENSION_NAMES:
    ci = bootstrap_ci([float(v) for v in dim_scores[dim]])
    dim_cis[dim] = ci
    med = sorted(dim_scores[dim])[len(dim_scores[dim]) // 2]
    print(f"{dim:<35s}  {ci['mean']:>6.3f}  [{ci['ci_lower']:.3f}, {ci['ci_upper']:.3f}]  {med:>6d}")

# Overall mean
overall_ci = bootstrap_ci(mean_scores)
print("─" * 75)
print(f"{'Overall mean (5 dimensions)':<35s}  {overall_ci['mean']:>6.3f}  [{overall_ci['ci_lower']:.3f}, {overall_ci['ci_upper']:.3f}]")

Dimension                              Mean                95% CI  Median
───────────────────────────────────────────────────────────────────────────
factual_correctness                   4.657  [4.529, 4.775]       5
completeness                          4.392  [4.255, 4.525]       5
grounding_faithfulness                4.750  [4.647, 4.843]       5
citation_usefulness                   4.324  [4.211, 4.431]       4
compliance_assessment_quality         4.299  [4.147, 4.446]       5
───────────────────────────────────────────────────────────────────────────
Overall mean (5 dimensions)           4.484  [4.375, 4.587]


### 6.2 Judge Score Distribution per Dimension

In [13]:
# --- Score distribution table ---
print(f"{'Dimension':<35s}  {'1':>5s} {'2':>5s} {'3':>5s} {'4':>5s} {'5':>5s}")
print("─" * 65)
for dim in DIMENSION_NAMES:
    counts = Counter(dim_scores[dim])
    row = "  ".join(f"{counts.get(i, 0):>3d}" for i in range(1, 6))
    print(f"{dim:<35s}  {row}")

Dimension                                1     2     3     4     5
─────────────────────────────────────────────────────────────────
factual_correctness                    3   11    7   11  172
completeness                           1   13   26   29  135
grounding_faithfulness                 0    9    7   10  178
citation_usefulness                    1    7   16   81   99
compliance_assessment_quality          4   20   17   33  130


### 6.3 Primary Weakness Distribution

The judge classifies the single most significant weakness of each answer
into one of: `none`, `unsupported_claim`, `incomplete_answer`,
`wrong_compliance_level`, `weak_citation_support`, `factual_error`.

In [14]:
weakness_counts = Counter(s["judge"].primary_weakness for s in rq3_samples)
print(f"{'Weakness Category':<30s}  {'Count':>5s}  {'%':>6s}")
print("─" * 45)
for weakness, count in weakness_counts.most_common():
    pct = 100 * count / len(rq3_samples)
    print(f"{weakness:<30s}  {count:>5d}  {pct:>5.1f}%")

Weakness Category               Count       %
─────────────────────────────────────────────
none                              123   60.3%
incomplete_answer                  41   20.1%
wrong_compliance_level             20    9.8%
factual_error                      13    6.4%
weak_citation_support               4    2.0%
unsupported_claim                   3    1.5%


## 7. Manual Audit Calibration Subset

Generate a stratified 20-sample subset for human calibration of the
LLM judge. The subset is stratified by:
- ESRS standard (ensure broad coverage across 12 standards)
- Compliance level (Low / Medium / High coverage)
- Retrieval quality tier (mix of high and lower recall samples)

Output: a CSV template with columns for each dimension (1-5) that the
researcher fills in manually. **No API calls required.**

In [15]:
import random

# --- Stratified sampling for manual audit ---
random.seed(42)
AUDIT_SIZE = 20

# Group samples by ESRS standard
by_standard = defaultdict(list)
for i, s in enumerate(rq3_samples):
    by_standard[s["esrs_standard"]].append(i)

# Select at least 1 from each standard, then fill remaining by level balance
audit_indices = set()
standards = sorted(by_standard.keys())

# Round 1: one per standard (12 standards → 12 samples)
for std in standards:
    candidates = by_standard[std]
    # Prefer variety in gold levels
    random.shuffle(candidates)
    audit_indices.add(candidates[0])

# Round 2: fill remaining 8 with level-balanced selection
remaining = [i for i in range(len(rq3_samples)) if i not in audit_indices]
random.shuffle(remaining)
by_level = defaultdict(list)
for i in remaining:
    by_level[rq3_samples[i]["gold_level"]].append(i)

# Fill evenly across levels
level_cycle = ["Low", "Medium", "High"]
level_idx = 0
while len(audit_indices) < AUDIT_SIZE:
    level = level_cycle[level_idx % 3]
    if by_level[level]:
        audit_indices.add(by_level[level].pop(0))
    level_idx += 1
    # Safety: break if all exhausted
    if not any(by_level[l] for l in level_cycle):
        break

audit_indices = sorted(audit_indices)
audit_samples = [rq3_samples[i] for i in audit_indices]

print(f"Audit subset size: {len(audit_samples)}")
print(f"Standards covered: {len(set(s['esrs_standard'] for s in audit_samples))} / {len(standards)}")
print(f"Level distribution: {dict(Counter(s['gold_level'] for s in audit_samples))}")

Audit subset size: 20
Standards covered: 11 / 11
Level distribution: {'High': 12, 'Medium': 4, 'Low': 4}


In [16]:
# --- Export audit CSV template ---
audit_rows = []
for s in audit_samples:
    j = s["judge"]
    audit_rows.append({
        "question_id": s["question_id"],
        "company": s["company"],
        "esrs_standard": s["esrs_standard"],
        "gold_level": s["gold_level"],
        "pred_level": s["pred_level"],
        # Judge scores for comparison
        "judge_factual_correctness": j.factual_correctness,
        "judge_completeness": j.completeness,
        "judge_grounding_faithfulness": j.grounding_faithfulness,
        "judge_citation_usefulness": j.citation_usefulness,
        "judge_compliance_assessment_quality": j.compliance_assessment_quality,
        # Human columns (to be filled manually)
        "human_factual_correctness": "",
        "human_completeness": "",
        "human_grounding_faithfulness": "",
        "human_citation_usefulness": "",
        "human_compliance_assessment_quality": "",
        # Reference: gold answer and generated answer (for the reviewer)
        "gold_answer_excerpt": s["gold_answer"][:300],
        "generated_answer_excerpt": s["gen_direct_answer"][:300],
        "judge_explanation": j.judge_explanation,
    })

audit_df = pd.DataFrame(audit_rows)
audit_csv_path = RQ3_RESULTS_DIR / "rq3_manual_audit_template.csv"
audit_df.to_csv(audit_csv_path, index=False, encoding="utf-8")
print(f"Audit CSV exported: {audit_csv_path}")
print(f"Columns: {list(audit_df.columns)}")

Audit CSV exported: /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq3/rq3_manual_audit_template.csv
Columns: ['question_id', 'company', 'esrs_standard', 'gold_level', 'pred_level', 'judge_factual_correctness', 'judge_completeness', 'judge_grounding_faithfulness', 'judge_citation_usefulness', 'judge_compliance_assessment_quality', 'human_factual_correctness', 'human_completeness', 'human_grounding_faithfulness', 'human_citation_usefulness', 'human_compliance_assessment_quality', 'gold_answer_excerpt', 'generated_answer_excerpt', 'judge_explanation']


## 8. Error Analysis (placeholder — implemented after full 204-sample run)

The following analyses will be populated after the full evaluation run:

1. **Per-ESRS-standard compliance accuracy table**
2. **Per-ESRS-standard judge-score table**
3. **Retrieval-generation gap analysis** (Recall@15 >= 0.8 but grounding <= 2/5)
4. **Compliance level confusion matrix** (3x3: predicted x gold)
5. **Failure-type table** from judge primary_weakness classifications

These cells are intentionally left as placeholders.

In [17]:
# --- Persist all RQ3 sample results for later analysis ---
rq3_rows = []
for s in rq3_samples:
    j = s["judge"]
    rq3_rows.append({
        "question_id": s["question_id"],
        "company": s["company"],
        "esrs_standard": s["esrs_standard"],
        "esrs_disclosure_id": s["esrs_disclosure_id"],
        "gold_level": s["gold_level"],
        "pred_level": s["pred_level"],
        "exact_match": s["gold_level"] == s["pred_level"],
        "ordinal_error": ordinal_error(s["gold_level"], s["pred_level"]),
        "cite_precision": s.get("cite_precision", None),
        "cite_recall": s.get("cite_recall", None),
        "cite_f1": s.get("cite_f1", None),
        "recall_at_15": s.get("recall_at_15", None),
        # Judge dimensions
        "factual_correctness": j.factual_correctness,
        "completeness": j.completeness,
        "grounding_faithfulness": j.grounding_faithfulness,
        "citation_usefulness": j.citation_usefulness,
        "compliance_assessment_quality": j.compliance_assessment_quality,
        "judge_mean_score": j.mean_score,
        "primary_weakness": j.primary_weakness,
        "judge_explanation": j.judge_explanation,
    })

rq3_df = pd.DataFrame(rq3_rows)
rq3_csv_path = RQ3_RESULTS_DIR / "rq3_sample_results.csv"
rq3_df.to_csv(rq3_csv_path, index=False, encoding="utf-8")
print(f"RQ3 sample results saved: {rq3_csv_path} ({len(rq3_df)} rows)")

RQ3 sample results saved: /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq3/rq3_sample_results.csv (204 rows)


## 9. Summary

Consolidate all RQ3 findings into a single summary JSON and print
the key results.

In [18]:
# --- Build and persist summary ---
summary = {
    "rq3_condition": RQ3_CONDITION,
    "esrs_context_helps": ESRS_CONTEXT_HELPS,
    "n_samples": len(rq3_samples),
    "n_valid_compliance": len(valid_pairs),
    "deterministic_metrics": {
        k: v for k, v in det_metrics.items()
        if k not in ("confusion_matrix", "per_label")
    },
    "deterministic_bootstrap_ci": {
        name: {"mean": ci["mean"], "ci_lower": ci["ci_lower"], "ci_upper": ci["ci_upper"]}
        for name, ci in boot_results.items()
    },
    "citation_metrics": {
        "precision": cite_prec_ci,
        "recall": cite_rec_ci,
        "f1": cite_f1_ci,
    },
    "judge_dimension_scores": {
        dim: {"mean": dim_cis[dim]["mean"], "ci_lower": dim_cis[dim]["ci_lower"],
              "ci_upper": dim_cis[dim]["ci_upper"]}
        for dim in DIMENSION_NAMES
    },
    "judge_overall_mean": {
        "mean": overall_ci["mean"],
        "ci_lower": overall_ci["ci_lower"],
        "ci_upper": overall_ci["ci_upper"],
    },
    "weakness_distribution": dict(weakness_counts),
    "confusion_matrix": det_metrics.get("confusion_matrix", []),
    "per_label_metrics": det_metrics.get("per_label", {}),
}

summary_path = RQ3_RESULTS_DIR / "rq3_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(f"Summary saved: {summary_path}")

# --- Print final summary ---
print("\n" + "=" * 72)
print("RQ3 RESULTS — Generation Quality Characterisation")
print("=" * 72)
print(f"\nSamples evaluated: {len(rq3_samples)}")
print(f"ESRS context injection benefit (RQ2): {'Yes' if ESRS_CONTEXT_HELPS else 'Null result'}")
print(f"\nA. DETERMINISTIC COMPLIANCE METRICS")
print(f"   Macro-F1:           {det_metrics['macro_f1']:.4f}")
print(f"   Balanced accuracy:  {det_metrics['balanced_accuracy']:.4f}")
print(f"   Exact accuracy:     {det_metrics['exact_accuracy']:.4f}")
print(f"   Weighted kappa:     {det_metrics['weighted_kappa']:.4f}")
print(f"   Off-by-one:         {det_metrics['off_by_one_accuracy']:.4f}")
print(f"\nB. LLM JUDGE SCORES (1-5 scale)")
for dim in DIMENSION_NAMES:
    ci = dim_cis[dim]
    print(f"   {dim:<35s}  {ci['mean']:.3f}  [{ci['ci_lower']:.3f}, {ci['ci_upper']:.3f}]")
print(f"   {'Overall mean':<35s}  {overall_ci['mean']:.3f}  [{overall_ci['ci_lower']:.3f}, {overall_ci['ci_upper']:.3f}]")
print(f"\nC. CITATION PAGE METRICS")
print(f"   Precision: {cite_prec_ci['mean']:.4f}, Recall: {cite_rec_ci['mean']:.4f}, F1: {cite_f1_ci['mean']:.4f}")

Summary saved: /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq3/rq3_summary.json

RQ3 RESULTS — Generation Quality Characterisation

Samples evaluated: 204
ESRS context injection benefit (RQ2): Null result

A. DETERMINISTIC COMPLIANCE METRICS
   Macro-F1:           0.5451
   Balanced accuracy:  0.5589
   Exact accuracy:     0.6912
   Weighted kappa:     0.4075
   Off-by-one:         0.9608

B. LLM JUDGE SCORES (1-5 scale)
   factual_correctness                  4.657  [4.529, 4.775]
   completeness                         4.392  [4.255, 4.525]
   grounding_faithfulness               4.750  [4.647, 4.843]
   citation_usefulness                  4.324  [4.211, 4.431]
   compliance_assessment_quality        4.299  [4.147, 4.446]
   Overall mean                         4.484  [4.375, 4.587]

C. CITATION PAGE METRICS
   Precision: 0.1217, Recall: 0.2614, F1: 0.1498
